In [ ]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

# SLAC XSIF to Bmad conversion

In [ ]:
# Patch in the slac2bmad package
import sys
sys.path.append('python')

In [ ]:
from slac2bmad.xsif import prepare_xsif, remove_comment_blocks, replace_set, replace_set_commands, fix_matrix, expand_names, fix_names, unfold_comments, fold_comments
from slac2bmad.desplit import desplit_eles, desplit_ele
from slac2bmad.replace import replace_element, replace_eles
from slac2bmad.bmad import finalize_bmad

from glob import glob
import shutil

import subprocess
import json
import os

# Remove Comment blocks

# Change Set commands

In [ ]:
replace_set('SET,  afa, afa, 1a')

#replace_set2('QUM1,   K1=')

# Expand names, correct matrix element syntax

In [ ]:
fix_matrix('RM(3,4)')    

In [ ]:
expand_names('  afa APER BLMO')

In [ ]:
fix_names(['sfafasfa safa APER RM(1,2)'])

# Folding and unfolding comments

In [ ]:
def test():
    L0 = ['123\n', '123    !comment\n', '   \n', '  !simple comment\n', '123!456#789\n']
    L1 = unfold_comments(L0)
    L2 = fold_comments(L1)
    print(L0)
    print(L1)
    print(L2)
test()

# Desplitting (in Bmad)

In [ ]:
line0 = 'qsx16_full: line = (qsx16, xcsx16, ycsx16, qsx16)'    
line1 = 'qsx16_full: line = (qsx16,  qsx16a)'  
line2 = 'qsx16_full: line = (qsx16)'  
print(desplit_ele(line2))

In [ ]:
desplit_eles(['fafaa', line0])

# Custom element replacements (in Bmad)

In [ ]:
NEWELES = {}

NEWELES['umasxh'] = """
!------- SXR Undulator -------
my_umasxh_k = 5.0
umasxh: wiggler, 
        type = "VGHPU",
        L_period = 0.039, 
        n_period = 87, 
        b_max = my_umasxh_k * 2*pi*m_electron / (c_light * 0.039), 
        L = 87*0.039, 
        ds_step = 0.039*10
        
umasxh[L] = umasxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""

NEWELES['umahxh'] = """
!------- HXR Undulator -------
my_umahxh_k = 2.0
umahxh: wiggler, 
        type = "HGVPU",
        L_period = 0.026, 
        n_period = 129, 
        b_max = my_umahxh_k * 2*pi*m_electron / (c_light * 0.026), 
        L = 129*0.026, 
        tilt=pi/2,
        ds_step = 0.026*10
        
umahxh[L] = umahxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """




NEWELES['pssxh'] = """
!------- SXR Phase Shifter -------
!
! B_max = 2pi/lambda * sqrt(2*PHASE_INTEGRAL / L)
! 
pssxh_phase_integral = 3814e-9  !T^2 m^3, maximum, from: T^2mm^3 (180-3814)
pssxh_L        = 0.0825   ! m 
pssxh_L_period = 0.075 ! m 
pssxh: wiggler, type = "phase shifter", 
    L = pssxh_L,
    b_max = 2*pi / pssxh_L_period * sqrt(2 * pssxh_phase_integral / pssxh_L  ),
    n_period = 1
pssxh[L] = pssxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""



NEWELES['pshxh'] = """
!------- HXR Phase Shifter -------
!
! B_max = 2pi/lambda * sqrt(2*PHASE_INTEGRAL / L)
! 
pshxh_phase_integral = 490e-9  !T^2 m^3, maximum, from: T^2mm^3 (80-490)
pshxh_L        = 0.0495 ! m 
pshxh_L_period = 0.045 ! m 
pshxh: wiggler, type = "phase shifter", 
    L = pshxh_L,
    b_max = 2*pi / pshxh_L_period * sqrt(2 * pshxh_phase_integral / pshxh_L  ),
    n_period = 1
pshxh[L] = pshxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""



#-----------------
# XLEAP-II wigglers
NEWELES['umxl1h'] = """
!------- XLEAP-II wigglers -------
umxl0h: wiggler, 
        type = "LCLS-I",
        L_period = 0.55, 
        n_period = 6, 
        b_max = 0, ! = K * 2*pi*m_electron / (c_light * 0.55), 
        L = 6*0.55
        !ds_step = 0.55*10
                
umxl0h[L] = umxl0h[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------

umxl1h: umxl0h

"""

# Inherit from umxl0h
NEWELES['umxl2h'] = """
umxl2h: umxl0h
"""

NEWELES['umxl3h'] = """
umxl3h: umxl0h
"""

NEWELES['umxl4h'] = """
umxl4h: umxl0h
"""

# This needs to be extended
NEWELES['duqxl'] = """
! Extend to account for real WIGGLER elements for XLEAP
duqxl: drift, L = 0.2166 + 0.03
"""



In [ ]:
# CU only replacements


CU_NEWELES = {}

CU_NEWELES['lh_und'] = """
!------- Laser Heater Undulator for Copper Linac -------
my_lh_und_k = 1.38523
lh_und: wiggler, 
        type = "laser_heater_undulator",
        L_period = 0.054, 
        n_period = 10, 
        b_max = my_lh_und_k * 2*pi*m_electron / (c_light * 0.054), 
         L = 10*0.054 ! Was: 0.506263, 
        ds_step = 0.054
        
lh_und[L] = lh_und[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """

CU_NEWELES['dh03a'] = """
! Shorten so that lh_und has an integer number of poles
dh03a: drift, l = 0.09290825 - ( 10*0.054 - 0.506263 ) /2, type = "CSR"
"""

CU_NEWELES['dh03b'] = """
! Shorten so that lh_und has an integer number of poles
dh03b: drift, l = 0.08401830- ( 10*0.054 - 0.506263 ) /2, type = "CSR"
"""

# Add these repalcements
CU_LINAC_REPLACEMENTS = json.load(open('good_cu_linac_replacements.json'))
for name, replace in CU_LINAC_REPLACEMENTS.items():
    CU_NEWELES[name.lower()+'_full'] = replace

print(CU_NEWELES.keys())


In [ ]:
# SC Only replacements

SC_NEWELES = {}

SC_NEWELES['umhtr'] = """
!------- Laser Heater Undulator for SC Linac -------
my_umhtr_k = 0.960143

umhtr: wiggler, 
        type = "laser_heater_undulator",
        L_period = 0.054, 
        n_period = 10, 
        b_max = my_umhtr_k * 2*pi*m_electron / (c_light * 0.054), 
        L = 10*0.054 ! Was: 0.506263, 
        ds_step = 0.054
        
umhtr[L] = umhtr[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """
# 
SC_NEWELES['dh02c'] = """
! Shorten so that umhtr has an integer number of poles
dh02c: drift, l = 0.2795065 - ( 10*0.054 - 0.506263 ) /2 , type = "CSR" !0.297036
"""

SC_NEWELES['dh02d'] = """
! Shorten so that umhtr has an integer number of poles
dh02d: drift, l = 0.2724707 - ( 10*0.054 - 0.506263 ) /2, type = "CSR" !0.2900002
"""





# Add these repalcements
SC_LINAC_REPLACEMENTS = json.load(open('good_sc_linac_replacements.json'))
for name, replace in SC_LINAC_REPLACEMENTS.items():
    SC_NEWELES[name.lower()+'_full'] = replace

    
#    
SC_NEWELES['cavl015_full'] = """
!---------------------
! CAVL015 LCAVITY
CAVL015_full: line = (CAVL015)
! contains zero length elements:
    CSP01[superimpose] = T
    CSP01[ref] = CAVL015
    CSP01[ref_origin] = beginning
    CSP01[offset] = 0.659221562
"""

    
print(SC_NEWELES.keys())


In [ ]:
print("!---------------------\n! CAVL015 LCAVITY\nCAVL015_full: line = (CAVL015)\n! contains zero length elements:\n    CSP01[superimpose] = T\n    CSP01[ref] = CAVL015\n    CSP01[ref_origin] = beginning\n    CSP01[offset] = 0.659221562\n\n")

In [ ]:
BENDS_TO_DESPLIT = [
    # cu_spec
    'bxs',
    # htr, ...
    'bxh1', 'bxh2', 'bxh3', 'bxh4',
    'bx01', 'bx02', 
    'bx11', 'bx12', 'bx13', 'bx14',
    'bx21', 'bx22', 'bx23', 'bx24',
    'bxkik',
    'bcx311', 'bcx312', 'bcx313', 'bcx314', 
    'by1', 'by2',
    'bx31', 'bx32', 
    'wig1h', 'wig2h', 'wig3h',
    'bcx321', 'bcx322', 'bcx323', 'bcx324', 
    'bykik1', 'bykik2',
    'bcx351', 'bcx352', 'bcx353', 'bcx354', 
    'bx35', 'bx36',
    'bcx361', 'bcx362', 'bcx363', 'bcx364',
    'bcxhs1', 'bcxhs2', 'bcxhs3', 'bcxhs4', 
    'bydsh',
    'byd1', 'byd2', 'byd3',
    # SXR:
    'brcusdc1', 'brcusdc2',
    'bkrcus',
    'blrcus',
    'bycus1', 'bycus2',
    'brcus1',
    'bcx31b1', 'bcx31b2', 'bcx31b3', 'bcx31b4',
    'bx31b', 'bx32b',
    'bykik1s', 'bykik2s',
    'bcx32b1', 'bcx32b2', 'bcx32b3', 'bcx32b4',
    'by1b', 'by2b',
    'bcxxl1', 'bcxxl2', 'bcxxl3', 'bcxxl4',
    'bcxss1', 'bcxss2', 'bcxss3', 'bcxss4', 
    'bydss',
    'byd1b', 'byd2b', 'byd3b',
    # sc_sxr
    'bcxh1', 'bcxh2', 'bcxh3', 'bcxh4',
    'bcx11', 'bcx12', 'bcx13', 'bcx14',
    'bcx21', 'bcx22', 'bcx23', 'bcx24',
    'bcxdlu1', 'bcxdlu2', 'bcxdlu3', 'bcxdlu4',
    'brb1', 'brb2',
    'bcxdld1', 'bcxdld2', 'bcxdld3', 'bcxdld4',
    'bkysp0s', 'bkysp1s', 'bkysp2s', 'bkysp3s', 'bkysp4s', 'bkysp5s',
    'blxsps',
    'bysp1s', 'bysp2s',
    'bxsp1s', 'bxsp2s', 'bxsp3s',
    # sc_hxr
    'bkysp0h', 'bkysp1h', 'bkysp2h', 'bkysp3h', 'bkysp4h', 'bkysp5h', 
    'blxsph',
    'bysp1h', 'bysp2h',
    'brsp1h', 'brsp2h',
    'bxsp1h',
    # sc_dasel
    'bkrdas1', 'bkrdas2', 'bkrdas3', 'bkrdas4', 'bkrdas5', 'bkrdas6',
    'blrdas',
    'brdas1', 'brdas2',
    'bram1', 
    'b11', 'b12', 'b13', 'b14', 'b15', 'b16',
    'b21', 'b22', 'b23', 'b24', 'b25', 'b26',
    # sc_diag0
    'bkrdg0', 'blrdg0', 'bxdg0', 'bydg0'
    
    
]
def desplit_bend_line(name):
    return f'{name}_full: line = ({name})'

BEND_REPLACEMENTS = {}
for name in BENDS_TO_DESPLIT:
    BEND_REPLACEMENTS[name+'_full'] = desplit_bend_line(name)
NEWELES.update(BEND_REPLACEMENTS )

In [ ]:
def all_replacements(master_file):
    dat = {}
    dat.update(NEWELES)
    if master_file.startswith('CU_'):
        print('CU replacements')
        dat.update(CU_NEWELES)
        return dat
    elif master_file.startswith('SC_'):
        print('SC replacements')
        dat.update(SC_NEWELES)
        return dat
    else:
        raise 
#all_replacements('CU_')        

# Full conversion

In [ ]:
#prepare_xsif('LTU.xsif')

In [ ]:
#translate_xsif_to_bmad('LTU.xsif')

In [ ]:
#with open('LTU.bmad') as f:
#    lines = f.readlines()

In [ ]:
#lines2 = replace_eles(lines, NEWELES)

In [ ]:
#NEWELES.keys()

In [ ]:

#finalize_bmad('UND.bmad', replacements=NEWELES)            

# Convert all

In [ ]:
!mkdir temp

In [ ]:
# Clean
!rm *xsif *bmad *digested*

In [ ]:
!pwd

In [ ]:
!cp /Users/chrisonian/Code/GitHub/lcls-lattice/mad/*xsif .

In [ ]:
XSIF_FILES=[f for f in os.listdir() if f.endswith('.xsif')]
for f in XSIF_FILES:
    prepare_xsif(f, save=False)

In [ ]:
!mv *xsif temp/

In [ ]:
CU_MASTERS = [f for f in os.listdir('../../mad') if f.startswith('CU_')]
SC_MASTERS = [f for f in os.listdir('../../mad') if f.startswith('SC_')]
CU_MASTERS, SC_MASTERS

In [ ]:
TEMPDIR = './temp/'
WORKDIR = './work/'

In [ ]:
!mkdir {TEMPDIR}
!mkdir {WORKDIR}

# Process 

In [ ]:
DEST = os.path.expandvars('$LCLS_LATTICE/bmad/master/')

In [ ]:
def process_master(master):
    
    print(f'Converting {master}')
    
    shutil.copytree(TEMPDIR, WORKDIR, dirs_exist_ok=True)
    
    # New method
    SCRIPT = f'cd work;python $ACC_ROOT_DIR/util_programs/mad_to_bmad/mad8_to_bmad.py --no_prepend_vars -f {master}'

    res = subprocess.run(SCRIPT, shell=True, cwd=WORKDIR)
    
    assert res.returncode == 0
    
    BMAD_FILES=glob(WORKDIR+'/*bmad')

    REPLACEMENTS = all_replacements(master)

    for f in BMAD_FILES:
        finalize_bmad(f, replacements=REPLACEMENTS, verbose=False)   
    
    print(f'    Copying all to {DEST}')
    for f in BMAD_FILES:
        #print(f'copying {f} to {DEST}')
        shutil.copy(f, DEST)
    
    
process_master('CU_HXR.xsif')

In [ ]:
for m in CU_MASTERS:
    process_master(m)

In [ ]:
for m in SC_MASTERS:
    process_master(m)

# Final cleanup

In [ ]:
!rm -r {TEMPDIR}
!rm -r {WORKDIR}